# So sánh các bộ tối ưu hóa (Optimizers) trong MLP từ Scratch

Notebook này minh họa chi tiết cấu trúc thuật toán và hiệu năng thực nghiệm của 4 bộ tối ưu phổ biến trong học sâu:
1. **Stochastic Gradient Descent (SGD)**
2. **SGD với Động lượng (Momentum)**
3. **RMSprop (Root Mean Squared Propagation)**
4. **Adam (Adaptive Moment Estimation)**

Mô hình sử dụng là một mạng nơ-ron truyền thẳng **MultiLayer Perceptron (MLP)** được hiện thực từ đầu bằng **NumPy**, huấn luyện trên tập dữ liệu phân loại phi tuyến **Half Moons**.

In [ ]:
# Bước 1: Import các thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Đã import thành công các thư viện!")

In [ ]:
# Bước 2: Định nghĩa mô hình MLP và thuật toán cập nhật của các Optimizers từ Scratch

class MLPClassifierWithOptimizers:
    """Mô hình mạng MLP hỗ trợ nhiều bộ tối ưu hóa khác nhau."""
    
    def __init__(
        self,
        layer_sizes: list[int],
        learning_rate: float = 0.01,
        epochs: int = 1000,
        optimizer: str = "adam",
        beta1: float = 0.9,
        beta2: float = 0.999,
        epsilon: float = 1e-8,
    ):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.epochs = epochs
        self.optimizer = optimizer.lower()
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        
        # 1. Khởi tạo trọng số (Weights) và độ chệch (Biases) dùng He Initialization
        self.W = []
        self.b = []
        for i in range(1, len(layer_sizes)):
            limit = np.sqrt(2.0 / layer_sizes[i - 1])
            self.W.append(np.random.randn(layer_sizes[i], layer_sizes[i - 1]) * limit)
            self.b.append(np.zeros((layer_sizes[i], 1)))
            
        # 2. Khởi tạo trạng thái lịch sử của các bộ tối ưu hóa (v: moment bậc 2/vận tốc, m: moment bậc 1)
        self.v_W = [np.zeros_like(w) for w in self.W]
        self.v_b = [np.zeros_like(bias) for bias in self.b]
        self.m_W = [np.zeros_like(w) for w in self.W]
        self.m_b = [np.zeros_like(bias) for bias in self.b]
        
        self.loss_history = []
        self.accuracy_history = []
        
    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
        
    @staticmethod
    def _sigmoid_derivative(a: np.ndarray) -> np.ndarray:
        return a * (1.0 - a)
        
    def _forward(self, X: np.ndarray) -> tuple[list[np.ndarray], list[np.ndarray]]:
        """Lan truyền tiến qua các lớp."""
        a_values = [X]
        z_values = []
        a = X
        for W_l, b_l in zip(self.W, self.b):
            z = W_l @ a + b_l
            a = self._sigmoid(z)
            z_values.append(z)
            a_values.append(a)
        return a_values, z_values
        
    def _backward(self, a_values: list[np.ndarray], y: np.ndarray) -> tuple[list[np.ndarray], list[np.ndarray]]:
        """Lan truyền ngược sai số tính Gradient (dW, db) dựa trên Chain Rule."""
        n_samples = y.shape[1]
        dW = []
        db = []
        
        # Tính lỗi ở lớp đầu ra (Output layer delta)
        a_out = a_values[-1]
        delta = (a_out - y) * self._sigmoid_derivative(a_out)
        
        num_layers = len(self.W)
        for l in reversed(range(num_layers)):
            a_prev = a_values[l]
            dW_l = (1.0 / n_samples) * (delta @ a_prev.T)
            db_l = (1.0 / n_samples) * np.sum(delta, axis=1, keepdims=True)
            
            dW.insert(0, dW_l)
            db.insert(0, db_l)
            
            if l > 0:
                delta = (self.W[l].T @ delta) * self._sigmoid_derivative(a_prev)
                
        return dW, db
        
    def _update_parameters(self, dW: list[np.ndarray], db: list[np.ndarray], t: int) -> None:
        """Cập nhật trọng số theo thuật toán tối ưu đã chọn."""
        for l in range(len(self.W)):
            if self.optimizer == "sgd":
                # SGD thuần: cập nhật trực tiếp
                self.W[l] -= self.lr * dW[l]
                self.b[l] -= self.lr * db[l]
                
            elif self.optimizer == "momentum":
                # Momentum: cập nhật tích lũy hướng vận tốc v
                self.v_W[l] = self.beta1 * self.v_W[l] + (1 - self.beta1) * dW[l]
                self.v_b[l] = self.beta1 * self.v_b[l] + (1 - self.beta1) * db[l]
                self.W[l] -= self.lr * self.v_W[l]
                self.b[l] -= self.lr * self.v_b[l]
                
            elif self.optimizer == "rmsprop":
                # RMSprop: Thích ứng learning rate dựa trên EMA bình phương gradient
                self.v_W[l] = self.beta2 * self.v_W[l] + (1 - self.beta2) * (dW[l] ** 2)
                self.v_b[l] = self.beta2 * self.v_b[l] + (1 - self.beta2) * (db[l] ** 2)
                self.W[l] -= self.lr * dW[l] / (np.sqrt(self.v_W[l]) + self.epsilon)
                self.b[l] -= self.lr * db[l] / (np.sqrt(self.v_b[l]) + self.epsilon)
                
            elif self.optimizer == "adam":
                # Adam: Kết hợp Moment bậc 1 (m) và bậc 2 (v) cùng Hiệu chỉnh độ lệch (Bias Correction)
                self.m_W[l] = self.beta1 * self.m_W[l] + (1 - self.beta1) * dW[l]
                self.m_b[l] = self.beta1 * self.m_b[l] + (1 - self.beta1) * db[l]
                
                self.v_W[l] = self.beta2 * self.v_W[l] + (1 - self.beta2) * (dW[l] ** 2)
                self.v_b[l] = self.beta2 * self.v_b[l] + (1 - self.beta2) * (db[l] ** 2)
                
                # Hiệu chỉnh độ lệch khởi đầu bằng 0
                m_W_hat = self.m_W[l] / (1.0 - self.beta1 ** t)
                m_b_hat = self.m_b[l] / (1.0 - self.beta1 ** t)
                v_W_hat = self.v_W[l] / (1.0 - self.beta2 ** t)
                v_b_hat = self.v_b[l] / (1.0 - self.beta2 ** t)
                
                self.W[l] -= self.lr * m_W_hat / (np.sqrt(v_W_hat) + self.epsilon)
                self.b[l] -= self.lr * m_b_hat / (np.sqrt(v_b_hat) + self.epsilon)
                
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MLPClassifierWithOptimizers":
        """Huấn luyện mạng nơ-ron."""
        X_input = X.T
        y_input = y.reshape(1, -1) if len(y.shape) == 1 else y.T
        
        self.loss_history = []
        self.accuracy_history = []
        
        for epoch in range(1, self.epochs + 1):
            # 1. Forward Pass
            a_values, _ = self._forward(X_input)
            y_hat = a_values[-1]
            
            # 2. Tính Loss (MSE)
            loss = 0.5 * np.mean(np.sum((y_hat - y_input) ** 2, axis=0))
            self.loss_history.append(loss)
            
            # 3. Tính Accuracy
            preds = (y_hat >= 0.5).astype(int)
            acc = float(np.mean(preds == y_input))
            self.accuracy_history.append(acc)
            
            # 4. Backward Pass (tính gradient)
            dW, db = self._backward(a_values, y_input)
            
            # 5. Cập nhật tham số theo bước thời gian t = epoch
            self._update_parameters(dW, db, epoch)
            
            # In kết quả định kỳ
            if epoch == 1 or epoch % 200 == 0 or epoch == self.epochs:
                print(f"Epoch {epoch:4d}/{self.epochs:4d} | Loss: {loss:.6f} | Train Acc: {acc:.2%}")
                
        return self
        
    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        X_input = X.T
        a_values, _ = self._forward(X_input)
        probs = a_values[-1].T
        return (probs >= threshold).astype(int).ravel()

print("Đã định nghĩa thành công lớp MLP Classifier!")

## Huấn luyện thử nghiệm so sánh 4 Optimizers

In [ ]:
# Bước 3: Tạo và phân tách tập dữ liệu
np.random.seed(42)
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Chuẩn hóa đặc trưng đầu vào
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

In [ ]:
# Bước 4: Thiết lập cấu hình và tiến hành huấn luyện so sánh

layer_sizes = [2, 10, 5, 1]
epochs = 1500

optimizers_config = {
    "SGD": {"optimizer": "sgd", "lr": 0.1},
    "Momentum": {"optimizer": "momentum", "lr": 0.1, "beta1": 0.9},
    "RMSprop": {"optimizer": "rmsprop", "lr": 0.01, "beta2": 0.99},
    "Adam": {"optimizer": "adam", "lr": 0.01, "beta1": 0.9, "beta2": 0.999},
}

results = {}

print("=" * 65)
print(" BẮT ĐẦU HUẤN LUYỆN SO SÁNH CÁC OPTIMIZERS")
print("=" * 65)

for name, config in optimizers_config.items():
    print(f"\n>>> Đang chạy bộ tối ưu: {name}...")
    mlp = MLPClassifierWithOptimizers(
        layer_sizes=layer_sizes,
        learning_rate=config["lr"],
        epochs=epochs,
        optimizer=config["optimizer"],
        beta1=config.get("beta1", 0.9),
        beta2=config.get("beta2", 0.999),
    )
    mlp.fit(X_train, y_train)
    
    # Đánh giá trên tập test
    test_preds = mlp.predict(X_test)
    test_acc = float(np.mean(test_preds == y_test))
    print(f"--> KẾT QUẢ {name}: Test Accuracy = {test_acc:.2%} | Final Loss = {mlp.loss_history[-1]:.6f}")
    
    results[name] = {
        "loss_history": mlp.loss_history,
        "accuracy_history": mlp.accuracy_history,
        "test_accuracy": test_acc,
    }

In [ ]:
# Bước 5: Trực quan hóa đồ thị so sánh Loss và Accuracy
plt.figure(figsize=(14, 6))

# Đồ thị Loss
plt.subplot(1, 2, 1)
for name, data in results.items():
    plt.plot(data["loss_history"], label=f"{name} (Test Acc: {data['test_accuracy']:.2%})", linewidth=2.0)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("So sánh Lịch sử Loss (Hội tụ)")
plt.grid(True, alpha=0.3)
plt.legend()

# Đồ thị Accuracy
plt.subplot(1, 2, 2)
for name, data in results.items():
    plt.plot(data["accuracy_history"], label=name, linewidth=2.0)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("So sánh Lịch sử Accuracy (Huấn luyện)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Ví dụ Tính Toán Số Học Từng Bước của Adam (t = 1)

Để kiểm chứng và hiểu sâu sắc hoạt động của Adam bên dưới mã nguồn, phần dưới đây chạy một bài toán 2 chiều đơn giản với các số liệu thực tế tại bước lặp đầu tiên.

In [ ]:
# Thiết lập thông số theo ví dụ tính toán thủ công
theta = np.array([1.5, -2.0])
g1 = np.array([0.5, -1.2])

# Siêu tham số
beta1 = 0.9
beta2 = 0.999
eta = 0.1
epsilon = 1e-8

# Trạng thái ban đầu t = 0
m0 = np.array([0.0, 0.0])
v0 = np.array([0.0, 0.0])

print("=== BƯỚC TÍNH SỐ THỰC TẾ CHO ADAM (t = 1) ===")
print(f"Vector tham số khởi đầu theta_1: {theta}")
print(f"Vector Gradient tại bước 1 g_1:  {g1}")

# Bước 1: Tính Moment bậc 1 (m1)
m1 = beta1 * m0 + (1.0 - beta1) * g1
print(f"\n1. Moment bậc 1 (chưa hiệu chỉnh) m_1:\n   m_1 = {m1}")

# Bước 2: Tính Moment bậc 2 (v1)
v1 = beta2 * v0 + (1.0 - beta2) * (g1 ** 2)
print(f"2. Moment bậc 2 (chưa hiệu chỉnh) v_1:\n   v_1 = {v1}")

# Bước 3: Hiệu chỉnh độ lệch (Bias Correction) với t = 1
t = 1
m1_hat = m1 / (1.0 - beta1 ** t)
v1_hat = v1 / (1.0 - beta2 ** t)
print(f"3. Moment hiệu chỉnh độ lệch tại t = 1:\n   m1_hat = {m1_hat} (Khớp với g_1)\n   v1_hat = {v1_hat} (Khớp với g_1^2)")

# Bước 4: Hướng cập nhật chuẩn hóa thang đo (U1)
U1 = m1_hat / (np.sqrt(v1_hat) + epsilon)
print(f"4. Hướng cập nhật U_1:\n   U_1 = {U1} (Bằng dấu của gradient sign(g_1))")

# Bước 5: Cập nhật tham số thu được theta_2
theta_2 = theta - eta * U1
print(f"5. Vector tham số mới theta_2 sau cập nhật:\n   theta_2 = {theta_2} (Kỳ vọng: [1.4, -1.9])")

assert np.allclose(theta_2, np.array([1.4, -1.9])), "Lỗi: Kết quả tính toán không khớp!"